# AI4RAG William benchmark — Baseline (pre–PR #75)

Compare prompt templates on the **same** GAM search space and William benchmark (22 questions).

**Run order:** `ai4rag_baseline_experiment.ipynb` → `ai4rag_pr75_experiment.ipynb`.

**Inspect without re-run:** set `RUN_EXPERIMENT = False` in section 2 and run from section 4 onward (loads `results/{variant}/runs/<latest>/`).

**Prerequisites:** OGX `.env`, William data in `../lightrag/POC/challenge_data/`, ~10–20 min if running GAM.

**Branch:** `move-autorag-components-code-to-ai4rag`

## 1. Install ai4rag

Skip if inspecting a saved run only (`RUN_EXPERIMENT = False`).

In [18]:
# Install ai4rag from: move-autorag-components-code-to-ai4rag
!pip install --force-reinstall git+https://github.com/IBM/ai4rag.git@move-autorag-components-code-to-ai4rag --quiet
!pip install python-dotenv openai pandas docling docling-core --quiet

print("Installed ai4rag @ move-autorag-components-code-to-ai4rag")

Installed ai4rag @ move-autorag-components-code-to-ai4rag


## 2. Configuration

In [19]:
import importlib
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)

VARIANT = "baseline"
BRANCH = "move-autorag-components-code-to-ai4rag"
PR_URL = "https://github.com/IBM/ai4rag/tree/move-autorag-components-code-to-ai4rag"
EXPECT_PR75 = False

# --- execution controls ---
RUN_EXPERIMENT = True   # False = load latest saved run (no GAM / no OGX experiment)
RUN_ID = None           # e.g. '20260623_163137' or None for latest
RUN_LLM_JUDGE = True    # 220 API calls; set False to skip or when reloading saved run

eu.print_saved_run_index(VARIANT)

if RUN_EXPERIMENT:
    env_path = eu.load_dotenv_from_standard_paths()
    print(f'Env: {env_path or "environment variables"}')
    OGX_BASE_URL, OGX_API_KEY = eu.require_ogx_credentials()
    print(f'OGX: {OGX_BASE_URL}')
    try:
        import ai4rag
        print(f'ai4rag version: {getattr(ai4rag, "__version__", "unknown")}')
    except ImportError as exc:
        raise ImportError('Run the install cell first') from exc
else:
    print('RUN_EXPERIMENT=False — will load persisted results in section 4')

Saved experiment runs

baseline/ (0 runs, latest=None)
Env: ../lightrag/POC/.env
OGX: https://<OGX_BASE_URL>
ai4rag version: 0.8.0


## 3. Load William benchmark

Skipped when `RUN_EXPERIMENT = False`.

In [20]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("baseline")

if RUN_EXPERIMENT:
    benchmark_data, documents = eu.load_william_benchmark()
    benchmark_df = pd.DataFrame(benchmark_data)
    print(f'Questions: {len(benchmark_data)}, documents: {len(documents)}')
else:
    benchmark_data, documents, benchmark_df = [], [], pd.DataFrame()
    print('Skipped (loading saved run)')

Questions: 22, documents: 10


## 4. Run experiment or load saved run

Fresh run: GAM explores retrieval/chunking from ai4rag defaults. Reload: reads `results/{variant}/runs/<run_id>/` (summary, leaderboard, answers, patterns, prompts).

In [21]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("baseline")

prompt_info = None

if RUN_EXPERIMENT:
    from ai4rag.rag.foundation_models.ogx import OgxClient

    ogx_client = OgxClient(base_url=OGX_BASE_URL, api_key=OGX_API_KEY)
    docling_documents = eu.documents_to_docling(documents)
    print(f'Converted {len(docling_documents)} documents')

    experiment = eu.run_gam_experiment(
        docling_documents, benchmark_df, ogx_client, label='baseline',
    )
    experiment_results, all_answers = eu.extract_experiment_results(experiment, benchmark_data)
    eu.enrich_results_with_answer_quality(experiment_results, all_answers)
    if experiment.results:
        top = experiment.results.get_best_evaluations(k=1)[0]
        prompt_info = eu.print_prompt_verification(top, expect_pr75=EXPECT_PR75)
    print('Experiment complete')
else:
    bundle = eu.load_saved_run(VARIANT, run_id=RUN_ID)
    experiment_results = bundle['experiment_results']
    all_answers = bundle['all_answers']
    leaderboard_df = bundle['leaderboard_df']
    prompt_info = bundle.get('prompts')
    run_dir = bundle['run_dir']
    print(f'Loaded saved run: {run_dir}')
    eu.print_run_inspection(bundle)

   [info] evaluation: Evaluating the RAG Pattern 'Pattern10' response using UnitxtEvaluator.
   🔧 Pattern unknown created
System message (first 200 chars):
You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, ...

User template (first 300 chars):
{reference_documents}
[conversation]: {question}. Answer with no more than 150 words. If you cannot base your answer on the given document, please state that you do not have an answer. Respond exclusively in the language of the question, regardless of any other language used in the provided context....

Context template:
[document]: {document}


PR #75 feature checklist:
  ❌ strong grounding
  ❌ mandatory citations
  ❌ english enforcement
  ❌ numbered documents

✅ Prompt verification OK for baseline (0/4 PR features).
Experiment complete


## 5. LLM-as-a-Judge

Skipped if answers already include `llm_judge` from a saved run.

In [22]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("baseline")

has_judge = all_answers and all('llm_judge' in a for a in all_answers)

if RUN_LLM_JUDGE and all_answers and not has_judge:
    if not RUN_EXPERIMENT:
        raise RuntimeError('LLM judge needs OGX — set RUN_EXPERIMENT=True or use a saved run that already has llm_judge')
    judge_fn = eu.create_llm_judge_fn(OGX_BASE_URL, OGX_API_KEY)
    eu.run_llm_judge_on_answers(all_answers, judge_fn)
    eu.attach_llm_judge_averages(experiment_results, all_answers)
    print('LLM-as-a-Judge complete')
elif has_judge:
    eu.attach_llm_judge_averages(experiment_results, all_answers)
    print('Using llm_judge scores from saved run')
else:
    print('Skipped LLM-as-a-Judge')

0.80
LLM-as-a-Judge complete


## 6. Leaderboard

In [27]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("baseline")

if experiment_results:
    leaderboard_df = eu.build_leaderboard_df(experiment_results)
elif 'leaderboard_df' not in globals() or leaderboard_df is None or getattr(leaderboard_df, 'empty', True):
    raise RuntimeError('Run section 4 first (no experiment_results or leaderboard_df)')
else:
    leaderboard_df = eu.coerce_leaderboard_df(leaderboard_df)

display_df = eu.format_leaderboard_for_display(leaderboard_df)
print("AI4RAG BASELINE LEADERBOARD")
print('=' * 100)
print(display_df.to_string(index=False))

if not leaderboard_df.empty:
    run_summary = eu.summarize_run(experiment_results, all_answers)
    print('\nRun summary:')
    for k, v in run_summary.items():
        print(f'  {k}: {v:.1%}')

AI4RAG BASELINE LEADERBOARD
  Pattern Retrieval  Window  Chunks Chunking Chunk Size Faithfulness Answer Correctness LLM Judge Citation Rate Multilingual % Combined  Final Score
 Pattern1    window       1       3      N/A        N/A        43.3%              35.6%     72.7%          0.0%          40.9%    39.5%       0.4329
Pattern10    window       1       3      N/A        N/A        39.1%              31.2%     76.4%          0.0%          59.1%    35.2%       0.3913
 Pattern2    window       5       5      N/A        N/A        35.3%              28.2%     78.2%          0.0%          68.2%    31.8%       0.3530
 Pattern9    window       1       3      N/A        N/A        34.9%              27.5%     74.5%          0.0%          63.6%    31.2%       0.3493
 Pattern6    window       1       5      N/A        N/A        34.3%              26.9%     75.5%          0.0%          63.6%    30.6%       0.3430
 Pattern4    window       1       5      N/A        N/A        34.1%          

## 7. Save artifacts

Writes a full run folder:
`results/baseline/runs/<timestamp>/` with `summary.json`, `leaderboard.csv`, `answers.csv`, `patterns.json`, `prompts.json`.

Skipped when reloading (`RUN_EXPERIMENT = False`).

In [28]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("baseline")

if RUN_EXPERIMENT and experiment_results:
    run_dir = eu.save_experiment_artifacts(
        VARIANT,
        branch=BRANCH,
        pr_url=PR_URL,
        benchmark_data=benchmark_data,
        experiment_results=experiment_results,
        all_answers=all_answers,
        leaderboard_df=leaderboard_df,
        prompt_info=prompt_info if isinstance(prompt_info, dict) else None,
    )
    print(f'Saved run: {run_dir}')
    print('Files: summary.json, leaderboard.csv, answers.csv, patterns.json, prompts.json')
else:
    print('Skipped save (no new experiment run)')

Saved run: /Users/lcmielow/Documents/GitHub/prototypes/AutoRAG/comparative_analysis/autorag/results/baseline/runs/20260624_082605
Files: summary.json, leaderboard.csv, answers.csv, patterns.json, prompts.json


## 8. Inspect saved runs (offline)

Browse all runs or load a specific `RUN_ID` without re-running GAM.

In [29]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("baseline")

eu.print_saved_run_index(VARIANT)

# Uncomment to inspect a specific run:
# inspect = eu.load_saved_run(VARIANT, run_id='20260623_163137')
# eu.print_run_inspection(inspect)
# inspect['answers_df'].head(10)

Saved experiment runs

baseline/ (2 runs, latest=20260624_082605)
  20260624_082605  best_faith=43.3% ← latest
  20260624_080128  best_faith=43.3%
